In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 86.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=f0bcaddf3829fda1f9112ee06f4f864bdd17d768d7d75b40ed3d5c735210931c
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



1. Add Protocol Settings

In [6]:
backend = BasicSimulator()

# Protocol settings
N = 24                  # number of qubits Alice sends
TEST_FRACTION = 0.5     # proportion of sifted key publicly checked
ATTACK_THRESHOLD = 0.15 # if error rate is above this, report attack

2. Create Quantum Random Bit Generator

In [7]:
def quantum_random_bits(number_of_bits):
    # Generate random bits by measuring |+> = 1/sqrt(2)(|0> + |1>).
    circuit = QuantumCircuit(number_of_bits, number_of_bits)
    circuit.h(range(number_of_bits))
    circuit.measure(range(number_of_bits), range(number_of_bits))

    compiled = transpile(circuit, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    bitstring = list(result.get_counts().keys())[0]

    # Qiskit displays classical bits right-to-left, so reverse it for q0, q1, q2, ... order.
    return [int(bit) for bit in bitstring[::-1]]

3. Prepare Alice and Bob measure one Qubit

In [8]:
def prepare_and_measure(alice_bit, alice_basis, bob_basis):

    # Alice prepares one qubit and Bob measures it.
    # basis 0 = computational / Z basis: |0>, |1>
    # basis 1 = diagonal / X basis: |+>, |->

    circuit = QuantumCircuit(1, 1)

    # Alice prepares the bit in the chosen basis.
    if alice_bit == 1:
        circuit.x(0)
    if alice_basis == 1:
        circuit.h(0)

    # Bob measures in his chosen basis.
    if bob_basis == 1:
        circuit.h(0)
    circuit.measure(0, 0)

    compiled = transpile(circuit, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    return int(list(result.get_counts().keys())[0])

4. Alice chooses secret bits and bases

In [9]:
# Alice randomly chooses data bits and bases using quantum randomness.
alice_bits = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

print("Alice bits: ", alice_bits)
print("Alice bases:", alice_bases)

Alice bits:  [1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1]
Alice bases: [0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1]


5. Bob chooses bases and measures Alice's Qubits

In [10]:
# Bob randomly chooses measurement bases using quantum randomness.
bob_bases = quantum_random_bits(N)
bob_results = []

for i in range(N):
    bob_results.append(prepare_and_measure(alice_bits[i], alice_bases[i], bob_bases[i]))

print("Bob bases:  ", bob_bases)
print("Bob results:", bob_results)

Bob bases:   [1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0]
Bob results: [0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1]


6. Public comparison

In [11]:
matching_positions = []
for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        matching_positions.append(i)

sifted_alice_key = [alice_bits[i] for i in matching_positions]
sifted_bob_key = [bob_results[i] for i in matching_positions]

print("Matching positions:", matching_positions)
print("Alice sifted key: ", sifted_alice_key)
print("Bob sifted key:   ", sifted_bob_key)

Matching positions: [1, 3, 5, 8, 9, 11, 15, 16, 18, 19, 20]
Alice sifted key:  [1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1]
Bob sifted key:    [1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1]


7. Reveal test bits and calculate error rate

In [12]:
test_size = max(1, math.floor(len(sifted_alice_key) * TEST_FRACTION))
test_alice = sifted_alice_key[:test_size]
test_bob = sifted_bob_key[:test_size]

errors = 0
for i in range(test_size):
    if test_alice[i] != test_bob[i]:
        errors += 1

error_rate = errors / test_size
attack_detected = error_rate > ATTACK_THRESHOLD

print("Test bits revealed:", test_size)
print("Errors in test bits:", errors)
print("Error rate:", round(error_rate, 3))
print("Attack detected?", attack_detected)

Test bits revealed: 5
Errors in test bits: 0
Error rate: 0.0
Attack detected? False


8. Create final key

In [13]:
final_alice_key = sifted_alice_key[test_size:]
final_bob_key = sifted_bob_key[test_size:]

print("BB84 WITHOUT ATTACKER")
print("Qubits sent:", N)
print("Matching bases:", len(matching_positions))
print("Alice final key:", final_alice_key)
print("Bob final key:  ", final_bob_key)
print("Final keys match?", final_alice_key == final_bob_key)

BB84 WITHOUT ATTACKER
Qubits sent: 24
Matching bases: 11
Alice final key: [1, 1, 0, 1, 1, 1]
Bob final key:   [1, 1, 0, 1, 1, 1]
Final keys match? True
